# Nexus Pipeline: Kokoro TTS + WhisperX (Auto-Sync)
Este notebook lê os arquivos `.txt` do seu Google Drive, gera o áudio com Kokoro (TTS SOTA) e executa o WhisperX para gerar os timestamps word-level (`.json`), salvando tudo de volta no Drive.

In [ ]:
# @title 1. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 2. Instalar Dependências (Kokoro + WhisperX)
!apt-get update && apt-get install -y espeak-ng
!pip install -q kokoro soundfile numpy torch
!pip install -q git+https://github.com/m-bain/whisperx.git

In [ ]:
# @title 3. Executar Pipeline (TTS -> Align -> Save)
import os
import torch
import soundfile as sf
import numpy as np
import json
import whisperx
from kokoro import KPipeline

# Configurações de Diretório no Drive
TEXTS_DIR = "/content/drive/MyDrive/nexus_pipeline/texts"
AUDIO_READY_DIR = "/content/drive/MyDrive/nexus_pipeline/audio_ready"
os.makedirs(AUDIO_READY_DIR, exist_ok=True)

print("[Inicializando Modelos]")
# Kokoro Pipeline
k_pipeline = KPipeline(lang_code='a')
# WhisperX Pipeline
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"WhisperX usando device: {device}")
whisper_model = whisperx.load_model("base.en", device, compute_type="float16" if device == "cuda" else "int8")

for filename in os.listdir(TEXTS_DIR):
    if not filename.endswith(".txt"):
        continue
        
    scene_id = filename.replace(".txt", "")
    txt_path = os.path.join(TEXTS_DIR, filename)
    wav_path = os.path.join(AUDIO_READY_DIR, f"{scene_id}.wav")
    json_path = os.path.join(AUDIO_READY_DIR, f"{scene_id}_words.json")
    
    if os.path.exists(wav_path) and os.path.exists(json_path):
        print(f"[Pulo] Cena {scene_id} já processada.")
        continue
        
    print(f"\n--- Processando Cena: {scene_id} ---")
    with open(txt_path, 'r', encoding='utf-8') as f:
        text = f.read().strip()
        
    if not text:
        print("Texto vazio, pulando.")
        continue

    # 1. Geração de Áudio (Kokoro)
    print("1. Gerando TTS...")
    generator = k_pipeline(text, voice='af_heart', speed=1, split_pattern=r'\n+')
    all_audio = []
    for _, _, audio in generator:
        if audio is not None:
            all_audio.append(audio)
            
    if all_audio:
        final_audio = np.concatenate(all_audio)
        sf.write(wav_path, final_audio, 24000)
        print(f"Áudio salvo: {wav_path}")
    else:
        print("Falha na geração de áudio.")
        continue

    # 2. Transcrição e Alinhamento Forçado (WhisperX)
    print("2. Extraindo Timestamps (WhisperX)...")
    audio = whisperx.load_audio(wav_path)
    result = whisper_model.transcribe(audio, batch_size=16)
    
    model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
    result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)
    
    # Extrair lista plana de palavras
    words_list = []
    for segment in result["segments"]:
        for word in segment.get("words", []):
            if "start" in word:
                words_list.append({
                    "word": word["word"],
                    "start": round(word["start"], 3),
                    "end": round(word["end"], 3)
                })
                
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(words_list, f, indent=2)
    print(f"Timestamps salvos: {json_path}")

print("\n✅ Processamento do Colab Concluído! Arquivos salvos no Google Drive.")